In [40]:
import os

BASE_DIR = "ABSOLUTE_PATH_TO_THE_ROOT"
DATA_DIR = os.path.join(BASE_DIR, "data")
CODE_DIR = os.path.join(BASE_DIR, "code")
FC_DIR = os.path.join(BASE_DIR, 'models_save/fc')
UC_DIR = os.path.join(BASE_DIR, 'models_save/uc')

import sys

sys.path.append(CODE_DIR)

import os
import json
from collections import defaultdict
from typing import Tuple

import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance

from matplotlib import pyplot as plt
import matplotlib.gridspec as grid_spec

from tqdm.auto import tqdm

from loader.generator import DataGenerator
from config import TSDataConfig, TaskConfig
from main_utils import _init_fc
from omegaconf import DictConfig

from models.forcast.darts import SimpleDartsModel
from models.forcast.forcast_service import ForcastService
from models.forcast.forcast_base import FCPredictionData
from models.uncertainty.uc_service import UncertaintyService
from models.uncertainty.dist_match.tree import DistMatchQRF
from models.uncertainty.dist_match.utils import match_ks_stat
from utils.calc_torch import calc_residuals

In [41]:
def matcher(x1, x2):
    return match_ks_stat(x1, x2) < 0.1


def get_qrf(path: str) -> DistMatchQRF:
    qrf = DistMatchQRF(
        alpha=0.1,
        n_quantile_bins=10,
        feature_dim=-1,
        matcher=matcher,
        match_mask=None,
        n_trees=10,
        bagging_ratio=0.9,
        verbose=False,
    )
    qrf.load_trees(path)
    return qrf

In [42]:
TASK_CONFIG = DictConfig(
    {
        "task_type": "PI",
        "alpha": 0.1,
        "data_splits": [0.6, 0.15, 0.25],
        "fc_estimator_mode": "single",
        "global_norm": False,
        "add_config": None,
    }
)


def load_dataset(data_config: TSDataConfig, task_config: TaskConfig = None):
    task_config = task_config or TaskConfig(**TASK_CONFIG)
    return DataGenerator.get_data(
        data_config=data_config,
        task_config=TASK_CONFIG,
        replace_base_dir=DATA_DIR,
        X_norm_param=None,
        Y_norm_param=None,
        hydro_static_norm_param=None,
    )


def load_forecast_service(
    fc_model=None, model_config=None, data_config=None, task_config=None
):
    if fc_model is None:
        fc_model = SimpleDartsModel
        model_config = dict(
            model="darts-forest", model_params={"lags": 50, "lags_past_covariates": 50}
        )

    task_config = task_config or TASK_CONFIG

    return ForcastService(
        int_fc_model=lambda: fc_model(**model_config),
        task_config=task_config,
        model_config=model_config,
        data_config=data_config,
        persist_dir=FC_DIR,
    )

In [43]:
def get_residuals(forcast_service: ForcastService, data, is_calib: bool = False) -> Tuple[np.ndarray, np.ndarray]:
    data = forcast_service.prepare([data], forcast_service._task_config.alpha)[0]

    if is_calib:
        calib_data = UncertaintyService._map_to_calib_data(data)
        fc_result = forcast_service.predict(
            FCPredictionData(
                ts_id=calib_data.ts_id,
                X_past=calib_data.X_pre_calib,
                Y_past=calib_data.Y_pre_calib,
                X_step=calib_data.X_calib,
                step_offset=calib_data.step_offset,
            )
        )
        return calc_residuals(Y_hat=fc_result.point, Y=calib_data.Y_calib).numpy(), fc_result.point

    fc_result = forcast_service.predict(
        FCPredictionData(
            ts_id=data.ts_id,
            X_past=data.X_calib,
            Y_past=data.Y_calib,
            X_step=data.X_test,
            step_offset=data.test_step,
        )
    )
    return calc_residuals(Y_hat=fc_result.point, Y=data.Y_test).numpy(), fc_result.point

In [44]:
from scipy.stats import ks_2samp

def test_kernel_stability(residuals, window_size=5, n_trials=200):
    ratios = []
    
    for trial in range(n_trials):
        n = len(residuals) - window_size - 1
        n_samples = 100
        
        idx1 = np.random.choice(n, size=n_samples, replace=False)
        idx2 = np.random.choice(n, size=n_samples, replace=False)
        
        # Patches as (n_samples, window_size) arrays
        patches_P = np.array([residuals[i:i+window_size] for i in idx1])
        patches_Q = np.array([residuals[i:i+window_size] for i in idx2])
        
        next_P = np.array([residuals[i+window_size] for i in idx1])
        next_Q = np.array([residuals[i+window_size] for i in idx2])
        
        # D_KS(P̃, Q̃): average KS over each position in patch
        ks_per_position = []
        for pos in range(window_size):
            ks = ks_2samp(patches_P[:, pos], patches_Q[:, pos]).statistic
            ks_per_position.append(ks)
        
        d_patches = np.mean(ks_per_position)
        
        # D_KS(P̃K, Q̃K): KS for next residuals
        d_next = ks_2samp(next_P, next_Q).statistic
        
        if d_patches > 0.01:
            ratios.append({
                'patch_dist': d_patches,
                'next_dist': d_next,
                'ratio': d_next / d_patches
            })
    
    return ratios

In [45]:
def get_patch_target_ratios(data, forcast_service, percentiles, patch_size, is_calib=True, n_trials=100):
    residuals, _ = get_residuals(forcast_service, data, is_calib=is_calib)
    ratios = test_kernel_stability(residuals, patch_size, n_trials=n_trials)
    # Extract just the ratio values from the list of dicts
    ratio_values = [r['ratio'] for r in ratios]
    return np.percentile(ratio_values, percentiles), np.mean(ratio_values)

In [46]:
PATCHES = [5, 10, 25, 50, 100, 150]
DATA_MAP = {
    "Elec": (
        "electric",
        ["/some_base_dir/data/enbPI/electricity-normalized.csv"],
        0,
    ),
    "Solar": (
        "solar",
        ["/some_base_dir/data/enbPI/Solar_Atl_data_aligned.csv"],
        0,
    ),
    "Wind": (
        "wind",
        ["/some_base_dir/data/enbPI/Wind_Hackberry_Generation_2019_2020.csv"],
        0,
    ),
}

for data_type, (datatype, data_paths, file_idx) in DATA_MAP.items():
    data_config = DictConfig(
        {"dataset_type": datatype, "paths": data_paths, "add_config": None}
    )
    prepared = load_dataset(TSDataConfig(**data_config))
    forcast_service = load_forecast_service(data_config=data_config)
    ratios, mean_ratio = get_patch_target_ratios(
        prepared[file_idx],
        forcast_service,
        PATCHES
    )
    print(datatype, ratios, mean_ratio)

electric [1.04166667 1.3859852  1.50075758 1.70625   ] 1.0289291296728948
(8760, 8)
solar [0.97693531 1.54418985 1.8052381  2.07033469] 1.0500202594933468
wind [1.01983821 1.44513575 1.57127716 1.71119883] 1.043135469160663
